In [1]:
import json
import numpy as np
import re

class EncounterLibrary:
    def __init__(self, encounters=None):
        self.encounters = encounters

    @classmethod
    def from_json_file(cls, json_path):
        with open(json_path, 'r') as fin:
            json_dict = json.load(fin)
        return cls(json_dict)

    def __repr__(self):
        return f'{self.__dict__}'
    
    def get_encounters(self, book_path):
        return [e for e in self.encounters if re.match(book_path, e['book_path'])]

class MonsterDatabase:
    def __init__(self, monsters=None):
        self.monsters = monsters

    @classmethod
    def from_json_file(cls, json_path):
        with open(json_path, 'r') as fin:
            json_dict = json.load(fin)
        return cls(json_dict)

    def __repr__(self):
        return f'{self.__dict__}'
    
    def get_monster(self, id):
        for m in self.monsters:
            if m['id'] == id:
                return m
        return None


def player_character_xp_budget(pc_level, rules='2014'):
    """Returns the adventuring day XP budget for a single PC of the given level.
    """
    if rules == '2014':
        XP_BUDGET = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
    else:
        XP_BUDGET = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
    return XP_BUDGET[pc_level-1]

def player_character_xp_thresholds(pc_level, rules='2014'):
    """Returns the encounter XP thresholds for each encounter difficulty for a PC of the given level.
    """
    if rules == '2014':
        XP_THRESHOLDS = {
            'Trivial':[  0,  0,   0,   0,   0,   0,   0,   0,   0,   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
            'Easy':   [ 25, 50,  75, 125, 250, 300, 350, 450, 550, 600,  800, 1000, 1100, 1250, 1400, 1600, 2000, 2100, 2400, 2800], 
            'Medium': [ 50,100, 150, 250, 500, 600, 750, 900,1100,1200, 1600, 2000, 2200, 2500, 2800, 3200, 3900, 4200, 4900, 5700], 
            'Hard':   [ 75,150, 225, 375, 750, 900,1100,1400,1600,1900, 2400, 3000, 3400, 3800, 4300, 4800, 5900, 6300, 7300, 8500], 
            'Deadly': [100,200, 400, 500,1100,1400,1700,2100,2400,2800, 3600, 4500, 5100, 5700, 6400, 7200, 8800, 9500,10900,12700], 
            'Very Deadly':  [x/2 for x in [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]], 
        }
    else:
        XP_THRESHOLDS = {
            'Low':      [ 50,100,150,250, 500, 600, 750,1000,1300,1600,1900,2200,2600,2900,3300,3800, 4500, 5000, 5500, 6400],
            'Moderate': [ 75,150,225,375, 750,1000,1300,1700,2000,2300,2900,3700,4200,4900,5400,6100, 7200, 8700,10700,13200],
            'High':     [100,200,400,500,1100,1400,1700,2100,2600,3100,4100,4700,5400,6200,7800,9800,11700,14200,17200,22000],
        }
    pc_xps = {}
    for diff in XP_THRESHOLDS:
        pc_xps[diff] = XP_THRESHOLDS[diff][pc_level-1]
    return pc_xps

def party_xp_budget(levels, rules='2014'):
    """Calculate the adventuring day XP budget for a party of PCs with the given levels.
    """
    # calculates the XP budget for a party of PCs
    return sum([player_character_xp_budget(lvl, rules=rules) for lvl in levels])

def party_xp_thresholds(levels, rules='2014'):
    """calculates the XP thresholds for a party of PCs based on their levels and the rules set being used
    """
    party_xps = {}
    for lvl in levels:
        pc_xps = player_character_xp_thresholds(lvl, rules=rules)
        for diff, xp in pc_xps.items():
            party_xps[diff] = party_xps.get(diff, 0) + xp
    
    return party_xps

def encounter_multiplier_DMG(pc_count, npc_count):
    """Returns the encounter multiplier given by the 2014 DMG
    pc_count -- number of PCs in the encounter
    npc_count -- number of NPCs in the encounter
    """
    n_array = np.asarray([1,2,3,7,11,15])
    m_array = np.asarray([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0])
    i = 1 + n_array[n_array <= max(npc_count,1)].argmax()
    if pc_count >= 6:
        i -= 1
    elif pc_count <= 2:
        i += 1
    return m_array[i]

def encounter_xp_total(pc_levels, monster_xps):
    """Returns the total XP of all monsters in the encounter
    """
    return sum(monster_xps)

def encounter_adjusted_xp_total(pc_levels, monster_xps, rules='2014'):
    """Calculates the adjusted XP total for an encounter based on the number of PCs, monsters, and the monster's XP values.
    """
    xp_total = encounter_xp_total(pc_levels, monster_xps)
    if rules == '2014':
        em = encounter_multiplier_DMG(len(pc_levels), len(monster_xps))
    else:
        em = 1
    return em*xp_total

def encounter_difficulty(pc_thresholds, encounter_xp, rules='2014'):
    """Determines the encounter's difficulty category by comparing its XP value against the party's XP thresholds.
    """
    difficulties = list(pc_thresholds.keys())
    xp_values = list(pc_thresholds.values())
    indx = np.argsort(xp_values)
    
    if rules == '2014':
        difficulty = 'Trivial'
        for i in indx:
            if encounter_xp >= xp_values[i]:
                difficulty = difficulties[i]
    elif rules == '2024':
        difficulty = 'Very High'
        for i in reversed(indx):
            if encounter_xp <= xp_values[i]:
                difficulty = difficulties[i]
    return difficulty

def get_monster_xp_values(encounter):
    """Converts an encounter's `monsters` entry into a list of XP values, with each value corresponding to one monster in the encounter.
    """
    xps = []
    for m in encounter['monsters']:
        xps.extend(m[0]*[m[2]])
    encounter['monster_XPs'] = xps
    return encounter

In [2]:
# Construct data frame
import numpy as np
import pandas as pd

rules = '2014'
book_groups = {}

party_size = 5 # 4, 4, 4-5, ?, ?, 4-5, ?
book_groups['Tales from the Yawning Portal'] = [
    {
        'adventure': 'The Sunless Citadel',
        'party': party_size*[1],
        'book_path': r'.*; The Sunless Citadel.*; (\d|1\d|2[0123])\..*',
    },
    {
        'adventure': 'The Sunless Citadel',
        'party': party_size*[2],
        'book_path': r'.*; The Sunless Citadel.*; (2[456789]|3\d|4[01])\..*',
    },
    {
        'adventure': 'The Sunless Citadel',
        'party': party_size*[3],
        'book_path': r'.*; The Sunless Citadel.*; (4[23589]|5[0456])\..*',
    },
    {
        'adventure': 'The Forge of Fury',
        'party': party_size*[3],
        'book_path': r'.*; The Forge of Fury.*; ([13]|4 and 4a|[59]|1[0124])\..*',
    },
    {
        'adventure': 'The Forge of Fury',
        'party': party_size*[4],
        'book_path': r'.*; The Forge of Fury.*; (1[56789]|2[0146]|3[03]|3[679]|4[1235789]|52)\..*',
    },
    {
        'adventure': 'The Hidden Shrine of Tamoachan',
        'party': party_size*[5],
        'book_path': r'.*; The Hidden Shrine of Tamoachan.*; ([37]|1[138]|2[0235])\..*', # Lower Chambers
    },
    {
        'adventure': 'The Hidden Shrine of Tamoachan',
        'party': party_size*[6],
        'book_path': r'.*; The Hidden Shrine of Tamoachan.*; (28|3[035789])\..*', # First Tier
    },
    {
        'adventure': 'The Hidden Shrine of Tamoachan',
        'party': party_size*[7],
        'book_path': r'.*; The Hidden Shrine of Tamoachan.*; (4[0236789]|5[24])\..*', # Second Tier, Third Tier, Temple Ground
    },
    {
        'adventure': 'White Plume Mountain',
        'party': party_size*[8],
        'book_path': r'.*; White Plume Mountain.*; (([258]|1[027]|2[0467])\.|Escaping the Dungeon).*',
    },
    {
        'adventure': 'Dead in Thay',
        'party': party_size*[9],
        'book_path': r'.*; Dead in Thay.*; (Abyssal Prisons|Blood Pens).*', #  ([123568]|1[01345])\.,  (1[789]|2[01234])\.
    },
    {
        'adventure': 'Dead in Thay',
        'party': party_size*[10],
        'book_path': r'.*; Dead in Thay.*; (Masters’ Domain|Far Realm Cysts).*', # (2[6789]|3[01235])\., (3[6789]|4[124])\.
    },
    {
        'adventure': 'Dead in Thay',
        'party': party_size*[10],
        'book_path': r'.*; Dead in Thay.*; (Forests of Slaughter|Ooze Grottos).*', 
    },
    {
        'adventure': 'Dead in Thay',
        'party': party_size*[11],
        'book_path': r'.*; Dead in Thay.*; (Predator Pools|Golem Laboratories).*', 
    },
    {
        'adventure': 'Dead in Thay',
        'party': party_size*[11],
        'book_path': r'.*; Dead in Thay.*; (Temples of Extraction|The Phylactery Vault).*', 
    },
    {
        'adventure': 'Against the Giants',
        'party': party_size*[11],
        'book_path': r'.*; Against the Giants.*; Steading of the Hill Giant Chief; Locations on the Upper Level; (1|1B|[23457]|1[179]|2[12])\..*',
    },
    {
        'adventure': 'Against the Giants',
        'party': party_size*[11],
        'book_path': r'.*; Against the Giants.*; Steading of the Hill Giant Chief; Locations on the Dungeon Level; ([23]|4–8|1[56]|2[138]|30)\..*',
    },
    {
        'adventure': 'Against the Giants',
        'party': party_size*[12],
        'book_path': r'.*; Against the Giants.*; The Glacial Rift of the Frost Giant Jarl; Locations on the Upper Level; ([12478]|9–10|1[235]|16–19|2[2356789])\..*',
    },
    {
        'adventure': 'Against the Giants',
        'party': party_size*[12],
        'book_path': r'.*; Against the Giants.*; The Glacial Rift of the Frost Giant Jarl; Locations on the Lower Level; (2|4B|[5679]|1[013467]|18–19|21)\..*',
    },
    {
        'adventure': 'Against the Giants',
        'party': party_size*[13],
        'book_path': r'.*; Against the Giants.*; Hall of the Fire Giant King; Locations on the Entrance Level; ([123456789]|1[012]|12A|1[45789]|2[012345])\..*',
    },
    {
        'adventure': 'Against the Giants',
        'party': party_size*[13],
        'book_path': r'.*; Against the Giants.*; Hall of the Fire Giant King; Locations on the Second Level; ([234567]|Cell Complex|8|1[234567])\..*',
    },
    {
        'adventure': 'Against the Giants',
        'party': party_size*[13],
        'book_path': r'.*; Against the Giants.*; Hall of the Fire Giant King; Locations on the Third Level; ([12456789]|10|11–13|1[56789]|20)\..*',
    },
    {
        'adventure': 'Tomb of Horrors',
        'party': party_size*[14],
        'book_path': r'.*; Tomb of Horrors.*; (8|13|18A|19|2[125679]|30|33)\..*',
    },
]

party_size = 5 #?, 4-6, 4-6, 4-6, 4-6, 4-6, 4-6
book_groups['Ghosts of Saltmarsh'] = [
    {
        'adventure': 'The Sinister Secret of Saltmarsh',
        'party': party_size*[1],
        'book_path': r'.*; The Sinister Secret of Saltmarsh.*; The Haunted House.*',
    },
    {
        'adventure': 'The Sinister Secret of Saltmarsh',
        'party': party_size*[2],
        'book_path': r'.*; The Sinister Secret of Saltmarsh.*; The Sea Ghost.*',
    },
    {
        'adventure': 'Danger at Dunwater',
        'party': party_size*[3],
        'book_path': r'.*; Danger at Dunwater; .*',
    },
    {
        'adventure': 'Salvage Operation',
        'party': party_size*[4],
        'book_path': r'.*; Salvage Operation; .*',
    },
    {
        'adventure': 'Isle of the Abbey',
        'party': party_size*[5],
        'book_path': r'.*; Isle of the Abbey.*; (The Island|The Abbey Ruins|The Ruined Cellar).*',
    },
    {
        'adventure': 'Isle of the Abbey',
        'party': party_size*[6],
        'book_path': r'.*; Isle of the Abbey.*; The Winding Way.*',
    },
    {
        'adventure': 'The Final Enemy', # This one probably needs more reading
        'party': party_size*[7],
        'book_path': r'.*; The Final Enemy.*; (Patrols in the Fortress|Fortress Level [12]).*',
    },
    {
        'adventure': 'The Final Enemy',
        'party': party_size*[8],
        'book_path': r'.*; The Final Enemy.*; Fortress Level 3.*',
    },
    {
        'adventure': 'Tammeraut’s Fate',
        'party': party_size*[9],
        'book_path': r'.*; Tammeraut’s Fate.*; (Harpy Attack|Island Approach and First Floor|Second Floor|Upper Levels and Cellar).*',
    },
    {
        'adventure': 'Tammeraut’s Fate',
        'party': party_size*[10],
        'book_path': r'.*; Tammeraut’s Fate.*; (Scenario 2|The Wreck).*',
    },
    {
        'adventure': 'The Styes',
        'party': party_size*[11],
        'book_path': r'.*; The Styes.*; (Mr. Dory’s Warehouse|Temple of Tharizdun|Landgrave’s Folly).*', #?
    },
]

party_size = 5 # 4-6 all
book_groups['Candlekeep Mysteries'] = [
    {
        'adventure': 'The Joy of Extradimensional Spaces',
        'party': party_size*[1],
        'book_path': r'.*; The Joy of Extradimensional Spaces; .*',
    },
    {
        'adventure': 'Mazfroth’s Mighty Digressions',
        'party': party_size*[2],
        'book_path': r'.*; Mazfroth’s Mighty Digressions; .*',
    },
    {
        'adventure': 'Book of the Raven',
        'party': party_size*[3],
        'book_path': r'.*; Book of the Raven; .*',
    },
    {
        'adventure': 'A Deep and Creeping Darkness',
        'party': party_size*[4],
        'book_path': r'^.*; A Deep and Creeping Darkness; .*',
    },
    {
        'adventure': 'Shemshime’s Bedtime Rhyme',
        'party': party_size*[4],
        'book_path': r'^.*; Shemshime’s Bedtime Rhyme; .*',
    },
    {
        'adventure': 'The Price of Beauty',
        'party': party_size*[5],
        'book_path': r'^.*; The Price of Beauty; .*',
    },
    {
        'adventure': 'Book of Cylinders',
        'party': party_size*[6],
        'book_path': r'^.*; Book of Cylinders; .*',
    },
    {
        'adventure': 'Sarah of Yellowcrest Manor',
        'party': party_size*[7],
        'book_path': r'^.*; Sarah of Yellowcrest Manor; .*',
    },
    {
        'adventure': 'Lore of Lurue',
        'party': party_size*[8],
        'book_path': r'^.*; Lore of Lurue; .*',
    },
    {
        'adventure': 'Kandlekeep Dekonstruktion',
        'party': party_size*[9],
        'book_path': r'^.*; Kandlekeep Dekonstruktion; .*',
    },
    {
        'adventure': 'Zikran’s Zephyrean Tome',
        'party': party_size*[10],
        'book_path': r'^.*; Zikran’s Zephyrean Tome; .*',
    },
    {
        'adventure': 'The Curious Tale of Wisteria Vale',
        'party': party_size*[11],
        'book_path': r'^.*; The Curious Tale of Wisteria Vale; .*',
    },
    {
        'adventure': 'The Book of Inner Alchemy',
        'party': party_size*[12],
        'book_path': r'^.*; The Book of Inner Alchemy; .*',
    },
    {
        'adventure': 'The Canopic Being',
        'party': party_size*[13],
        'book_path': r'^.*; The Canopic Being; .*',
    },
    {
        'adventure': 'The Scrivener’s Tale',
        'party': party_size*[14],
        'book_path': r'^.*; The Scrivener’s Tale; .*', 
    },
    {
        'adventure': 'Alkazaar’s Appendix',
        'party': party_size*[15],
        'book_path': r'^.*; Alkazaar’s Appendix; .*',
    },
    {
        'adventure': 'Xanthoria',
        'party': party_size*[16],
        'book_path': r'^.*; Xanthoria; .*',
    },
]

party_size = 5 # 4-6 all
book_groups['Journeys through the Radiant Citadel'] = [
    {
        'adventure': 'Salted Legacy',
        'party': party_size*[1],
        'book_path': r'.*; Salted Legacy.*; Battle Prawn Challenge.*',
    },
    {
        'adventure': 'Salted Legacy',
        'party': party_size*[2],
        'book_path': r'.*; Salted Legacy.*; Confronting Kasem.*',
    },
    {
        'adventure': 'Written in Blood',
        'party': party_size*[3],
        'book_path': r'.*; Written in Blood; .*',
    },
    {
        'adventure': 'The Fiend of Hollow Mine.',
        'party': party_size*[4],
        'book_path': r'^.*; The Fiend of Hollow Mine; .*',
    },
    {
        'adventure': 'Wages of Vice',
        'party': party_size*[5],
        'book_path': r'^.*; Wages of Vice; .*',
    },
    {
        'adventure': 'Sins of Our Elders',
        'party': party_size*[6],
        'book_path': r'^.*; Sins of Our Elders; .*',
    },
    {
        'adventure': 'Gold for Fools and Princes',
        'party': party_size*[7],
        'book_path': r'^.*; Gold for Fools and Princes; .*',
    },
    {
        'adventure': 'Trail of Destruction',
        'party': party_size*[8],
        'book_path': r'^.*; Trail of Destruction; .*',
    },
    {
        'adventure': 'In the Mists of Manivarsha',
        'party': party_size*[9],
        'book_path': r'^.*; In the Mists of Manivarsha; .*',
    },
    {
        'adventure': 'Between Tangled Roots',
        'party': party_size*[10],
        'book_path': r'^.*; Between Tangled Roots; .*',
    },
    {
        'adventure': 'Shadow of the Sun',
        'party': party_size*[11],
        'book_path': r'^.*; Shadow of the Sun; .*',
    },
    {
        'adventure': 'The Nightsea’s Succor',
        'party': party_size*[12],
        'book_path': r'^.*; The Nightsea’s Succor; .*',
    },
    {
        'adventure': 'Buried Dynasty',
        'party': party_size*[13],
        'book_path': r'^.*; Buried Dynasty; .*',
    },
    {
        'adventure': 'Orchids of the Invisible Mountain',
        'party': party_size*[14],
        'book_path': r'^.*; Orchids of the Invisible Mountain; .*',
    },
]

party_size = 5 # 4-6 all
book_groups['Keys from the Golden Vault'] = [
    {
        'adventure': 'The Murkmire Malevolence',
        'party': party_size*[1],
        'book_path': r'^.*; The Murkmire Malevolence; .*',
    },
    {
        'adventure': 'The Stygian Gambit',
        'party': party_size*[2],
        'book_path': r'^.*; The Stygian Gambit; .*',
    },
    {
        'adventure': 'Reach for the Stars',
        'party': party_size*[3],
        'book_path': r'^.*; Reach for the Stars; .*',
    },
    {
        'adventure': 'Prisoner 13',
        'party': party_size*[4],
        'book_path': r'^.*; Prisoner 13; .*',
    },
    {
        'adventure': 'Tockworth’s Clockworks',
        'party': party_size*[5],
        'book_path': r'^.*; Tockworth’s Clockworks; .*',
    },
    {
        'adventure': 'Masterpiece Imbroglio',
        'party': party_size*[5],
        'book_path': r'^.*; Masterpiece Imbroglio; .*',
    },
    {
        'adventure': 'Axe from the Grave',
        'party': party_size*[6],
        'book_path': r'^.*; Axe from the Grave; .*',
    },
    {
        'adventure': 'Vidorant’s Vault',
        'party': party_size*[7],
        'book_path': r'^.*; Vidorant’s Vault; .*',
    },
    {
        'adventure': 'Shard of the Accursed',
        'party': party_size*[8],
        'book_path': r'^.*; Shard of the Accursed; .*',
    },
    {
        'adventure': 'Heart of Ashes',
        'party': party_size*[8],
        'book_path': r'^.*; Heart of Ashes; .*',
    },
    {
        'adventure': 'Affair on the Concordant Express',
        'party': party_size*[9],
        'book_path': r'^.*; Affair on the Concordant Express; .*',
    },
    {
        'adventure': 'Party at Paliset Hall',
        'party': party_size*[10],
        'book_path': r'^.*; Party at Paliset Hall; .*',
    },
    {
        'adventure': 'Fire and Darkness',
        'party': party_size*[11],
        'book_path': r'^.*; Fire and Darkness; .*',
    },
]

party_size = 5 # 4-6, 4-6, 4-6, 4-6, 4-6, 4-6
book_groups['Quests from the Infinite Staircase'] = [
    {
        'adventure': 'The Lost City',
        'party': party_size*[1],
        'book_path': r'.*; The Lost City.*; Ziggurat Locations, Tiers 1–3; B\d: .*',
    },
    {
        'adventure': 'The Lost City',
        'party': party_size*[2],
        'book_path': r'.*; The Lost City.*; Ziggurat Locations, Tiers 1–3; B\d\d: .*',
    },
    {
        'adventure': 'The Lost City',
        'party': party_size*[3],
        'book_path': r'.*; The Lost City.*; Ziggurat Locations, Tier [45]',
    },
    {
        'adventure': 'When a Star Falls',
        'party': party_size*[4],
        'book_path': r'.*; When a Star Falls.*; (Death on the Moors|Piyarz’s Shadow|Cernant Valley|Derro Lair)',
    },
    {
        'adventure': 'When a Star Falls',
        'party': party_size*[5],
        'book_path': r'.*; When a Star Falls.*; (Tower of the Heavens|Forge of the Kagu-Svirfneblin)',
    },
    {
        'adventure': 'Beyond the Crystal Cave',
        'party': party_size*[6],
        'book_path': r'.*; Beyond the Crystal Cave.*; (Cave of Echoes|Eternal Garden|Palace of Spires)',
    },
    {
        'adventure': 'Pharaoh',
        'party': party_size*[7],
        'book_path': r'.*; Pharaoh.*; (False Tomb|Maze of Mists)',
    },
    {
        'adventure': 'Pharaoh',
        'party': party_size*[8],
        'book_path': r'.*; Pharaoh.*; (Halls of the Upper Priesthood|Gauntlet|Tomb of Amun Sa)',
    },
    {
        'adventure': 'The Lost Caverns of Tsojcanth',
        'party': party_size*[9],
        'book_path': r'.*; The Lost Caverns of Tsojcanth.*; (Yatil Mountains|Lesser Caverns Locations)',
    },
    {
        'adventure': 'The Lost Caverns of Tsojcanth',
        'party': party_size*[10],
        'book_path': r'.*; The Lost Caverns of Tsojcanth.*; Greater Caverns Locations',
    },
    {
        'adventure': 'Expedition to the Barrier Peaks',
        'party': party_size*[11],
        'book_path': r'.*; Expedition to the Barrier Peaks.*; Spaceship (Encounters|Locations), Level [12]',
    },
    {
        'adventure': 'Expedition to the Barrier Peaks',
        'party': party_size*[12],
        'book_path': r'.*; Expedition to the Barrier Peaks.*; Spaceship Locations, Level [34]',
    },
]

party_size = 5 # 4-6 all
book_groups['Dragon Delves'] = [
    {
        'adventure': 'Death at Sunset',
        'party': party_size*[1],
        'book_path': r'.*; Chapter 1: Death at Sunset; Redwood Grove.*',
    },
    {
        'adventure': 'Death at Sunset',
        'party': party_size*[2],
        'book_path': r'.*; Chapter 1: Death at Sunset; Death-at-Sunset’s Lair.*',
    },
    {
        'adventure': 'Baker’s Doesn’t',
        'party': party_size*[3],
        'book_path': r'.*; Chapter 2: Baker’s Doesn’t; .*',
    },
    {
        'adventure': 'The Will of Orcus',
        'party': party_size*[4],
        'book_path': r'.*; Chapter 3: The Will of Orcus; .*',
    },
    {
        'adventure': 'For Whom the Void Calls',
        'party': party_size*[5],
        'book_path': r'.*; Chapter 4: For Whom the Void Calls; .*',
    },
    {
        'adventure': 'The Dragon of Najkir',
        'party': party_size*[7],
        'book_path': r'.*; Chapter 5: The Dragon of Najkir; .*',
    },
    {
        'adventure': 'The Forbidden Vale',
        'party': party_size*[9],
        'book_path': r'.*; Chapter 6: The Forbidden Vale; .*',
    },
    {
        'adventure': 'Before the Storm',
        'party': party_size*[10],
        'book_path': r'.*; Chapter 7: Before the Storm; .*',
    },
    {
        'adventure': 'Shivering Death',
        'party': party_size*[11],
        'book_path': r'.*; Chapter 8: Shivering Death; .*',
    },
    {
        'adventure': 'A Copper for a Song',
        'party': party_size*[12],
        'book_path': r'.*; Chapter 9: A Copper for a Song; .*',
    },
    {
        'adventure': 'Dragons of the Sandstone City',
        'party': party_size*[12],
        'book_path': r'.*; Chapter 10: Dragons of the Sandstone City; .*',
    },
]

party_size = 5 # 4-6
book_groups['Tyranny of Dragons'] = [
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[1],
        'book_path': r'.*; Greenest in Flames; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[2],
        'book_path': r'.*; Raiders’ Camp; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[3],
        'book_path': r'.*; Dragon Hatchery; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[4],
        'book_path': r'.*; On the Road; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[5],
        'book_path': r'.*; Construction Ahead; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[5],
        'book_path': r'.*; Castle Naerytar; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[6],
        'book_path': r'.*; Hunting Lodge; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[7],
        'book_path': r'.*; Castle in the Clouds; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[8],
        'book_path': r'.*; Death to the Wyrmspeakers; (Varram the White|Tomb of Diderius|Ss’tck’al).*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[9],
        'book_path': r'.*; The Sea of Moving Ice; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[10],
        'book_path': r'.*; The Cult Strikes Back; First Attack.*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[11],
        'book_path': r'.*; Death to the Wyrmspeakers; (Neronvain|The Misty Forest|Neronvain’s Stronghold).*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[12],
        'book_path': r'.*; The Cult Strikes Back; Second Attack.*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[13],
        'book_path': r'.*; Xonthal’s Tower; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[14],
        'book_path': r'.*; Mission to Thay; .*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[14],
        'book_path': r'.*; The Cult Strikes Back; Third Attack.*',
    },
    {
        'adventure': 'Tyranny of Dragons',
        'party': party_size*[15],
        'book_path': r'.*; Tiamat’s Return; .*',
    },
]

party_size = 5 # 4-6
book_groups["Storm King's Thunder"] = [
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[1],
        'book_path': r'.*; A Great Upheaval; Nightstone; \d+\..*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[2],
        'book_path': r'.*; A Great Upheaval; Nightstone; Special Events in Nightstone.*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[3],
        'book_path': r'.*; A Great Upheaval; Dripping Caves.*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[4],
        'book_path': r'.*; A Great Upheaval; Unfriendly Skies.*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[5],
        'book_path': r'.*; Rumblings; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[6],
        'book_path': r'.*; The Savage Frontier; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[7],
        'book_path': r'.*; The Chosen Path; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[8],
        'book_path': r'.*; Den of the Hill Giants; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[8],
        'book_path': r'.*; Canyon of the Stone Giants; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[8],
        'book_path': r'.*; Berg of the Frost Giants; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[8],
        'book_path': r'.*; Forge of the Fire Giants; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[8],
        'book_path': r'.*; Castle of the Cloud Giants; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[9],
        'book_path': r'.*; Hold of the Storm Giants; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[9],
        'book_path': r'.*; Caught in the Tentacles; .*',
    },
    {
        'adventure': "Storm King's Thunder",
        'party': party_size*[10],
        'book_path': r'.*; Doom of the Desert; .*',
    },
]

party_size = 5 # ?
book_groups["Waterdeep: Dungeon of the Mad Mage"] = [
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[5],
        'book_path': r'.*; Dungeon Level.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[6],
        'book_path': r'.*; Arcane Chambers.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[7],
        'book_path': r'.*; Sargauth Level.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[7],
        'book_path': r'.*; Skullport.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[8],
        'book_path': r'.*; Twisted Caverns.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[8],
        'book_path': r'.*; Wyllowwood.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[9],
        'book_path': r'.*; Lost Level.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[9],
        'book_path': r'.*; Maddgoth’s Castle.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[10],
        'book_path': r'.*; Slitherswamp.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[10],
        'book_path': r'.*; Dweomercore.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[11],
        'book_path': r'.*; Muiral’s Gauntlet.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[11],
        'book_path': r'.*; Troglodyte Warrens.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[12],
        'book_path': r'.*; Maze Level.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[12],
        'book_path': r'.*; Trobriand’s Graveyard.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[13],
        'book_path': r'.*; Arcturiadoom.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[13],
        'book_path': r'.*; Obstacle Course.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[14],
        'book_path': r'.*; Crystal Labyrinth.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[14],
        'book_path': r'.*; Seadeeps.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[15],
        'book_path': r'.*; Vanrakdoom.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[15],
        'book_path': r'.*; Caverns of Ooze.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[16],
        'book_path': r'.*; Runestone Caverns.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[16],
        'book_path': r'.*; Terminus Level.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[17],
        'book_path': r'.*; Shadowdusk Hold.*',
    },
    {
        'adventure': 'Waterdeep: Dungeon of the Mad Mage',
        'party': party_size*[17],
        'book_path': r'.*; Mad Wizard’s Lair.*',
    },
]

party_size = 5 # 4-6
book_groups['Icewind Dale: Rime of the Frostmaiden'] = [
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[1],
        'book_path': r'.*; Ten-Towns.*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[2],
        'book_path': r'.*; (Bremen|Bryn Shander|Caer-Konig|Targos|Termalaine).*', 
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[3],
        'book_path': r'.*; (Caer-Dineval|Dougan’s Hole|Easthaven|Good Mead|Lonelywood).*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[4],
        'book_path': r'.*; Icewind Dale.*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[4],
        'book_path': r'.*; Places of Interest \(continued\).*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[5],
        'book_path': r'.*; Sunblight.*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[6],
        'book_path': r'.*; Destruction’s Light.*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[7],
        'book_path': r'.*; Auril’s Abode.*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[8],
        'book_path': r'.*; Caves of Hunger.*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[9],
        'book_path': r'.*; Doom of Ythryn.*; (Necropolis Locations \(Y1-Y9\)|Dealing with the Arcane Brotherhood).*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[10],
        'book_path': r'.*; Doom of Ythryn.*; Necropolis Locations \(Y10-Y18\).*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[11],
        'book_path': r'.*; Doom of Ythryn.*; (Ythryn Encounters|Spire of Iriolarthas).*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[12],
        'book_path': r'.*; Doom of Ythryn.*; (Necropolis Locations \(Y20-Y29\)|Auril’s Wrath).*',
    },
    {
        'adventure': 'Icewind Dale: Rime of the Frostmaiden',
        'party': party_size*[12],
        'book_path': r'.*; Auril the Frostmaiden.*',
    },
]

party_size = 5 # 5
book_groups['Critical Role: Call of the Netherdeep'] = [
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[3],
        'book_path': r'.*; A Fateful Competition.*; .*',
    },
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[4],
        'book_path': r'.*; The Leave-Taking.*; .*',
    },
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[5],
        'book_path': r'.*; Bazzoxan.*; (No Time for Pleasantries|Gloomstalker Escape).*',
    },
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[6],
        'book_path': r'^.*; Bazzoxan.*; (Locations in the Betrayers’ Rise).*',
    },
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[7],
        'book_path': r'^.*; Faction Story Tracks.*; .*Mission [123]:.*',
    },
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[8],
        'book_path': r'^.*; The Drowned City.*; (Uncharted Waters|Cael Morrow Locations \(M1-M8\)).*',
    },
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[9],
        'book_path': r'^.*; The Drowned City.*; (Cael Morrow Locations \(M9-M17\)).*',
    },
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[10],
        'book_path': r'^.*; The Netherdeep; Grottoes of Regret.*',
    },
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[11],
        'book_path': r'^.*; The Netherdeep; (Vents of Fury|Chasm of Yearning).*',
    },
    {
        'adventure': 'Call of the Netherdeep',
        'party': party_size*[12],
        'book_path': r'^.*; The Heart of Despair; .*',
    },
]

party_size = 5 # 4-6
book_groups['Dragonlance: Shadow of the Dragon Queen'] = [
    #{
    #    'adventure': 'Shadow of the Dragon Queen',
    #    'party': party_size*[1],
    #    'book_path': r'.*; Prelude to War; .*',
    #},
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[2],
        'book_path': r'.*; When Home Burns; (Betrayal at High Hill).*',
    },
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[3],
        'book_path': r'.*; When Home Burns; (Back in Vogler|Sighting the Enemy|A Fateful Morning|Invasion of Vogler).*',
    },
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[4],
        'book_path': r'.*; Shadow of War; (The First Mission|Missions for Kalaman).*',
    },
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[5],
        'book_path': r'.*; Shadow of War; (Wheelwatch Outpost|Battle at Steel Springs|The Lord’s Arrival|Raided Catacombs).*',
    },
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[6],
        'book_path': r'^.*; The Northern Wastes; (Into the Wastes|Exploring the Wastes; [ABC]:).*',
    },
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[7],
        'book_path': r'^.*; The Northern Wastes; Exploring the Wastes; [DEFGHIJK]:.*',
    },
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[8],
        'book_path': r'^.*; City of Lost Names; (Path of Memories|Test of High Sorcery|Exploring the City|Occupied Mansion|Temple of Paladine).*',
    },
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[9],
        'book_path': r'^.*; City of Lost Names; (Threshold of the Heavens|The City Rises|Escaping the Enemy).*',
    },
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[10],
        'book_path': r'^.*; Siege of Kalaman; (A Hasty Retreat|Return to Kalaman|Day of Dread|Night of Terror|Battle of Kalaman).*',
    },
    {
        'adventure': 'Shadow of the Dragon Queen',
        'party': party_size*[11],
        'book_path': r'^.*; Siege of Kalaman; (The Flying Citadel|Dragon Army Rout).*',
    },
]

party_size = 5 # 4-6
book_groups['Phandelver and Below: The Shattered Obelisk'] = [
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[1],
        'book_path': r'.*; A Dangerous Journey; .*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[2],
        'book_path': r'.*; Trouble in Phandalin; .*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[3],
        'book_path': r'.*; The Spider’s Web; .*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[4],
        'book_path': r'^.*; Wave Echo Cave; .*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[5],
        'book_path': r'^.*; Paths of Peril; (Townmaster’s Plight|Stolen Shards).*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[6],
        'book_path': r'^.*; Paths of Peril; (Zorzula’s Rest|Indigo Sanctum).*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[7],
        'book_path': r'^.*; The Shattered Obelisk; (Return to Phandalin|Talhundereth).*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[8],
        'book_path': r'^.*; The Shattered Obelisk; (Crypt of the Talhund|Gibbet Crossing).*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[9],
        'book_path': r'^.*; Rifts in Reality; .*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[10],
        'book_path': r'^.*; Beyond a Lightless Star; (The Briny Maze).*',
    },
    {
        'adventure': 'Phandelver and Below: The Shattered Obelisk',
        'party': party_size*[11],
        'book_path': r'^.*; Beyond a Lightless Star; (The Endless Void|Ilvaash’s Refraction).*',
    },
]

party_size = 5 # 4-6
book_groups['Vecna: Eve of Ruin'] = [
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[10],
        'book_path': r'.*; Return from Neverdeath Graveyard; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[11],
        'book_path': r'.*; The Wizards Three; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[12],
        'book_path': r'.*; The Lambent Zenith’s Last Voyage; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[13],
        'book_path': r'^.*; The Ruined Colossus; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[14],
        'book_path': r'^.*; Death House; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[15],
        'book_path': r'^.*; Night of Blue Fire; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[16],
        'book_path': r'^.*; Tomb of Wayward Souls; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[17],
        'book_path': r'^.*; The Dragon Queen’s Pride; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[18],
        'book_path': r'^.*; The Betrayer Revealed; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[19],
        'book_path': r'^.*; The War of Pandesmos; .*',
    },
    {
        'adventure': 'Vecna: Eve of Ruin',
        'party': party_size*[20],
        'book_path': r'^.*; Eve of Ruin; .*',
    },
]

book_dict = {
    "Tales from the Yawning Portal": {'acronym': 'TftYP', 'file_name': 'tftyp.json'},
    "Ghosts of Saltmarsh": {'acronym': 'GoS', 'file_name': 'gos.json'},
    "Candlekeep Mysteries": {'acronym': 'CM', 'file_name': 'cm.json'},
    "Journeys through the Radiant Citadel": {'acronym': 'JttRC', 'file_name': 'jttrc.json'},
    "Keys from the Golden Vault": {'acronym': 'KftGV', 'file_name': 'kftgv.json'},
    "Quests from the Infinite Staircase": {'acronym': 'QftIS', 'file_name': 'qftis.json'},
    "Dragon Delves": {'acronym': 'DD', 'file_name': 'drde.json'},
    "Tyranny of Dragons": {'acronym': 'ToD', 'file_name': 'tod.json'},
    "Storm King's Thunder": {'acronym': 'SKT', 'file_name': 'skt.json'},
    "Waterdeep: Dungeon of the Mad Mage": {'acronym': 'W:DotMM', 'file_name': 'wdotmm.json'},
    "Icewind Dale: Rime of the Frostmaiden": {'acronym': 'ID:RotF', 'file_name': 'idrotf.json'},
    "Critical Role: Call of the Netherdeep": {'acronym': 'CR:CotN', 'file_name': 'cotn.json'},
    "Dragonlance: Shadow of the Dragon Queen": {'acronym': 'D:SotDQ', 'file_name': 'sotdq.json'},
    "Phandelver and Below: The Shattered Obelisk": {'acronym': 'PaB:TSO', 'file_name': 'pbtso.json'},
    "Vecna: Eve of Ruin": {'acronym': 'V:EoR', 'file_name': 'veor.json'},
}

book_categories = list(book_groups.keys())
book_acronym_categories = [v['acronym'] for v in book_dict.values()]

encounters = []
for book_name in book_groups:
    book_acronym = book_dict[book_name]['acronym']
    file_name = book_dict[book_name]['file_name']
    elib = EncounterLibrary().from_json_file(f'./{file_name}')
    for group in book_groups[book_name]:
        encs = elib.get_encounters(group['book_path'])
        for enc in encs:
            if enc['type'] != 'combat': continue
            try:
                enc = get_monster_xp_values(enc)
                enc['book_acronym'] = book_acronym
                enc['adventure'] = group['adventure']
                enc['rules'] = rules
                enc['n_monsters'] = len(enc['monster_XPs'])
                enc['XP_total'] = encounter_xp_total(group['party'], enc['monster_XPs'])
                enc['adj_XP_total'] = encounter_adjusted_xp_total(group['party'], enc['monster_XPs'], rules=rules)
                enc['party'] = group['party']
                enc['party_xp_thresholds'] = party_xp_thresholds(group['party'], rules=rules)
                enc['party_xp_budget'] = party_xp_budget(group['party'], rules=rules)
                enc['PC_XP_total'] = enc['XP_total']/len(group['party'])
                enc['difficulty'] = encounter_difficulty(enc['party_xp_thresholds'], enc['adj_XP_total'], rules=rules)
                encounters.append(enc)
            except:
                #print(f"{enc['book_path']}:")
                #print(f"  monsters: {enc['monsters']}")
                pass

if rules == '2014':
    difficulty_categories = ['Trivial','Easy','Medium','Hard', 'Deadly', 'Very Deadly']
else:
    difficulty_categories = ['Low','Moderate','High','Very High']

df = pd.DataFrame({
    'book': [e['book'] for e in encounters],
    'book_acronym': [e['book_acronym'] for e in encounters],
    'adventure': [e['adventure'] for e in encounters],
    'party': [e['party'] for e in encounters],
    'party_level': [np.mean(e['party']) for e in encounters],
    'monsters': [e['monsters'] for e in encounters],
    'n_monsters': [e['n_monsters'] for e in encounters],
    'XP_total': [e['XP_total'] for e in encounters],
    'adj_XP_total': [e['adj_XP_total'] for e in encounters],
    'PC_XP_total': [e['PC_XP_total'] for e in encounters],
    'party_XP_budget': [e['party_xp_budget'] for e in encounters],
    'difficulty': [e['difficulty'] for e in encounters],
})
df['book'] = df['book'].astype('category')
df['book'] = df['book'].cat.set_categories(book_categories, ordered=True)
df['book_acronym'] = df['book_acronym'].astype('category')
df['book_acronym'] = df['book_acronym'].cat.set_categories(book_acronym_categories, ordered=True)
df['difficulty'] = df['difficulty'].astype('category')
df['difficulty'] = df['difficulty'].cat.set_categories(difficulty_categories, ordered=True)
df['XP_ratio'] = 2*df['adj_XP_total']/df['party_XP_budget']
df['XP_mult'] = df['adj_XP_total']/df['XP_total']